# Day 65：毕业答辩（15分钟演示 + 10分钟企业评委答辩评分引擎与 Q&A 模拟）

> **🎙️ 讲师旁白**：欢迎来到《星辰数科 · 智枢企业级智能体平台》第六阶段收官日——**Day 65（`v1.0.0-day65`）**！本 Notebook 将在本地端到端实测：
> 1. **五维度加权评分标准（`DefenseRubric`）**：业务价值 20%、技术架构与 RAG/Agent 深度 30%、工程质量与可观测性 20%、演示流畅度 15%、评委问答表现 15%；
> 2. **多评委聚合评估与评分卡导出（`evaluate_defense`）**：模拟 3 位企业评委打分，输出加权总分、`S/A/B/C` 评级与 Markdown 评分卡；
> 3. **企业评委硬核题库匹配与应答诊断（`JudgeQuestionBank`）**：覆盖幻觉治理、高并发成本、权限隔离、HITL 超时、微调 vs RAG 选型。

## 1. 初始化评分引擎与五维度权重检查

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day65_dir = repo_root / "project_iterations" / "day_65"
if str(day65_dir) not in sys.path:
    sys.path.insert(0, str(day65_dir))

from zhishu_agent import (
    DefenseRubric,
    JudgeQuestionBank,
    RubricDimension,
    evaluate_defense,
)

rubric = DefenseRubric()
print("已加载模块路径:", day65_dir)
for dim in rubric.dimensions:
    print(f"- {dim.name} ({dim.key}): 权重 {int(dim.weight * 100)}%")

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_65
- 业务价值 (business_value): 权重 20%
- 技术架构与RAG/Agent深度 (architecture_depth): 权重 30%
- 工程质量与可观测性 (engineering_quality): 权重 20%
- 演示流畅度 (demo_fluency): 权重 15%
- 评委问答表现 (qa_performance): 权重 15%


## 2. 模拟 3 位企业评委联合打分与 Markdown 评分卡生成

In [2]:
judge_evaluations = [
    {
        "judge_name": "周明远",
        "judge_role": "外部企业 CTO",
        "scores": {
            "business_value": 91.0,
            "architecture_depth": 94.0,
            "engineering_quality": 90.0,
            "demo_fluency": 88.0,
            "qa_performance": 92.0,
        },
        "comment": "混合 RAG 与 HITL 闭环完整，并发成本路由思路清晰。",
    },
    {
        "judge_name": "沈清扬",
        "judge_role": "信息安全与合规总监",
        "scores": {
            "business_value": 89.0,
            "architecture_depth": 91.0,
            "engineering_quality": 92.0,
            "demo_fluency": 86.0,
            "qa_performance": 90.0,
        },
        "comment": "Pre-filtering ACL 隔离与 409 重复审批拦截满足内控要求。",
    },
    {
        "judge_name": "张工",
        "judge_role": "星辰数科 Tech Lead",
        "scores": {
            "business_value": 90.0,
            "architecture_depth": 93.0,
            "engineering_quality": 91.0,
            "demo_fluency": 87.0,
            "qa_performance": 89.0,
        },
        "comment": "全队协同交付稳定，建议补充百并发压测数据。",
    },
]

result = evaluate_defense(
    scores=judge_evaluations,
    rubric=rubric,
    project_name="智枢企业级智能体平台 (ZhiShu-Agent v1.0.0-day65)",
    team_name="林晓 / 张工 / 陈默 / 李婷 / 赵磊",
)
print(f"综合得分: {result.total_score:.2f} | 等级: {result.grade}")
print("\n" + result.to_markdown())

综合得分: 90.60 | 等级: S

# 毕业答辩评审评分卡 — 智枢企业级智能体平台 (ZhiShu-Agent v1.0.0-day65)

- **汇报团队**：林晓 / 张工 / 陈默 / 李婷 / 赵磊
- **综合加权总分**：**90.60 / 100.00**
- **答辩评级**：**S 级**

## 一、五维度加权得分明细

| 评分维度 | 权重 | 评委均分 | 加权贡献分 |
| --- | :---: | :---: | :---: |
| 业务价值 (`business_value`) | 20% | 90.00 | 18.00 |
| 技术架构与RAG/Agent深度 (`architecture_depth`) | 30% | 92.67 | 27.80 |
| 工程质量与可观测性 (`engineering_quality`) | 20% | 91.00 | 18.20 |
| 演示流畅度 (`demo_fluency`) | 15% | 87.00 | 13.05 |
| 评委问答表现 (`qa_performance`) | 15% | 90.33 | 13.55 |

## 二、评委独立打分与评语

- **周明远**（外部企业 CTO）：加权得分 **91.40** — 混合 RAG 与 HITL 闭环完整，并发成本路由思路清晰。
- **沈清扬**（信息安全与合规总监）：加权得分 **89.90** — Pre-filtering ACL 隔离与 409 重复审批拦截满足内控要求。
- **张工**（星辰数科 Tech Lead）：加权得分 **90.50** — 全队协同交付稳定，建议补充百并发压测数据。

## 三、核心优势亮点

- 技术架构与RAG/Agent深度（均分 92.7，权重 30%）
- 工程质量与可观测性（均分 91.0，权重 20%）
- 评委问答表现（均分 90.3，权重 15%）
- 业务价值（均分 90.0，权重 20%）
- 演示流畅度（均分 87.0，权重 15%）

## 四、后续改进计划（Action Items）

- [演示流畅度 | 均分 87.0] 精简背景铺垫至 2 分钟内，将核心 SSE 流式与 HITL 审批闭环提前至前 8 分钟演示。



## 3. 企业评委硬核题库匹配与回答覆盖度诊断

In [3]:
bank = JudgeQuestionBank()
print(f"题库共收录 {len(bank.list_questions())} 道企业级考题:")
for q in bank.list_questions():
    print(f"  [{q.question_id}] {q.category} ({q.judge_role})")

query = "如果审批人出差一直不点通过，HITL 挂起超时和重复点击批准怎么处理？"
matched_q, relevance = bank.match_question(query, top_k=1)[0]
print(f"\n提问匹配 -> [{matched_q.question_id}] {matched_q.category} (相关度={relevance})")

candidate_answer = (
    "我们将审批单持久化到数据库支持服务重启恢复，设置 TTL 超时自动过期并触发升级提醒，"
    "审批接口使用 CAS 校验仅允许 pending 流转，重复批准返回 409 Conflict 并结合幂等键防重。"
)
diag = bank.score_answer(matched_q.question_id, candidate_answer)
print(f"回答诊断 -> 命中采分点: {len(diag['matched_points'])}/{len(matched_q.key_points)} | "
      f"覆盖率: {diag['coverage_ratio']*100:.0f}% | 建议得分: {diag['suggested_score']}")

题库共收录 5 道企业级考题:
  [JQ-01] 幻觉治理 (AI 算法与质量评审专家)
  [JQ-02] 高并发成本 (CTO / FinOps 架构负责人)
  [JQ-03] 权限隔离 (信息安全与合规总监)
  [JQ-04] HITL超时 (业务运维与风控 VP)
  [JQ-05] 微调vs RAG选型 (首席 AI 架构师)

提问匹配 -> [JQ-04] HITL超时 (相关度=6.5)
回答诊断 -> 命中采分点: 2/4 | 覆盖率: 50% | 建议得分: 77.5
